# SVC — 오류 분석

공개된 공동 오류 분석 중 모델별 혼동쌍 및 점수/이웃 진단을 분리한 구현. 공동 geometry 및 개인별 그림은 원본 Notebook에서 확인.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("SVC_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("SVC", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
sbase=base
print("Local output:",root)

In [ ]:
wrong=np.flatnonzero(base!=y)
rows=pd.DataFrame({"true":np.array(labels)[y[wrong]],"pred":np.array(labels)[base[wrong]]})
rows.groupby(["true","pred"]).size().reset_index(name="count").sort_values("count",ascending=False).to_csv(root/"01_error_analysis"/"Confusions.csv",index=False)
scores=model.decision_function(Xv[wrong])
top=np.argsort(scores,axis=1)[:,::-1]
print("True class ranked second:",int((top[:,1]==y[wrong]).sum()),"/",len(wrong))
print(rows.groupby(["true","pred"]).size().sort_values(ascending=False).head(20))